# Concatenation Effect Analysis

**Research Questions:**
- **b_1**: What is the effect of concatenation of SMILES strings on the tokens learnt?
- **b_2**: Did concatenation have a smaller effect for the polymer dataset than the molecular dataset?

## Comparisons

### Question b_1 - Overall concatenation effect:
- **PI1M**: `PI1M_noconcat_5epoch` vs `PI1M_concat_5epoch`
- **MOSES**: `MOSES_noconcat_5epoch` vs `MOSES_concat_5epoch`

### Question b_2 - Differential effect by dataset:
- Compare magnitude of change for PI1M vs MOSES


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup
sns.set_style("whitegrid")
sns.set_context("talk")
colors = sns.color_palette("mako", 10)
plt.rcParams['figure.figsize'] = (14, 8)

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from analysis.utils.statistics import TokenStatistics, compare_token_distributions, compute_kl_divergence

# Load all statistics
stats_dir = project_root / 'analysis' / 'data' / 'statistics'
pi1m_noconcat = TokenStatistics.load(str(stats_dir / 'PI1M_noconcat_5epoch_stats.json'))
pi1m_concat = TokenStatistics.load(str(stats_dir / 'PI1M_concat_5epoch_stats.json'))
moses_noconcat = TokenStatistics.load(str(stats_dir / 'MOSES_noconcat_5epoch_stats.json'))
moses_concat = TokenStatistics.load(str(stats_dir / 'MOSES_concat_5epoch_stats.json'))

print("✓ Statistics loaded!")


/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


✓ Statistics loaded!


## 1. Effect of Concatenation on PI1M (Polymer)


In [2]:
comparison_pi1m = compare_token_distributions(
    pi1m_noconcat, pi1m_concat,
    label1="No Concat", label2="With Concat"
)

print("="*60)
print("PI1M: Effect of Concatenation")
print("="*60)
print(f"Token Overlap Jaccard: {comparison_pi1m['token_overlap']['jaccard_similarity']:.4f}")
print(f"Breakpoint Overlap Jaccard: {comparison_pi1m['breakpoint_overlap']['jaccard_similarity']:.4f}")
print(f"Mean Token Length Diff: {comparison_pi1m['length_comparison']['mean_diff']:.4f}")

kl_div_pi1m = compute_kl_divergence(pi1m_noconcat, pi1m_concat)
print(f"KL Divergence: {kl_div_pi1m:.4f}")


PI1M: Effect of Concatenation
Token Overlap Jaccard: 0.3514
Breakpoint Overlap Jaccard: 0.7556
Mean Token Length Diff: 0.4524
KL Divergence: 2.0670


## 2. Effect of Concatenation on MOSES (Molecular)


In [3]:
comparison_moses = compare_token_distributions(
    moses_noconcat, moses_concat,
    label1="No Concat", label2="With Concat"
)

print("="*60)
print("MOSES: Effect of Concatenation")
print("="*60)
print(f"Token Overlap Jaccard: {comparison_moses['token_overlap']['jaccard_similarity']:.4f}")
print(f"Breakpoint Overlap Jaccard: {comparison_moses['breakpoint_overlap']['jaccard_similarity']:.4f}")
print(f"Mean Token Length Diff: {comparison_moses['length_comparison']['mean_diff']:.4f}")

kl_div_moses = compute_kl_divergence(moses_noconcat, moses_concat)
print(f"KL Divergence: {kl_div_moses:.4f}")


MOSES: Effect of Concatenation
Token Overlap Jaccard: 0.4493
Breakpoint Overlap Jaccard: 0.9600
Mean Token Length Diff: 0.1269
KL Divergence: 1.3073


## 3. Compare Concatenation Effect: PI1M vs MOSES

Test hypothesis: polymer dataset less affected by concatenation than molecular dataset.


In [4]:
# Compare magnitude of changes
summary_data = {
    'Metric': [
        'Token Overlap Jaccard',
        'Breakpoint Overlap Jaccard',
        'Mean Token Length Diff',
        'KL Divergence',
    ],
    'PI1M (Polymer)': [
        f"{comparison_pi1m['token_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_pi1m['breakpoint_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_pi1m['length_comparison']['mean_diff']:.4f}",
        f"{kl_div_pi1m:.4f}",
    ],
    'MOSES (Molecular)': [
        f"{comparison_moses['token_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_moses['breakpoint_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_moses['length_comparison']['mean_diff']:.4f}",
        f"{kl_div_moses:.4f}",
    ],
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("Concatenation Effect: PI1M vs MOSES")
print("="*60)
print(summary_df.to_string(index=False))

print("\n" + "="*60)
print("Interpretation:")
print("="*60)
print("Higher Jaccard similarity = less change due to concatenation")
print("Lower KL divergence = less change due to concatenation")

# Save summary
summary_df.to_csv(project_root / 'analysis' / 'data' / 'concatenation_effect_summary.csv', index=False)
print("\n✓ Analysis complete!")



Concatenation Effect: PI1M vs MOSES
                    Metric PI1M (Polymer) MOSES (Molecular)
     Token Overlap Jaccard         0.3514            0.4493
Breakpoint Overlap Jaccard         0.7556            0.9600
    Mean Token Length Diff         0.4524            0.1269
             KL Divergence         2.0670            1.3073

Interpretation:
Higher Jaccard similarity = less change due to concatenation
Lower KL divergence = less change due to concatenation

✓ Analysis complete!
